# 09. Hàm mục tiêu Huber

Ridge là hàm bậc hai, nên Newton xong sau một vòng và backtracking nhận ngay bước đầy đủ.
Notebook này chạy cùng bốn thuật toán trên hàm mất mát Huber, nơi cả hai đặc quyền đó
biến mất. Chương 8 của báo cáo.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
pd.set_option("display.width", 160)

In [2]:
from src.dataset import load_processed, SWEEP
from src.huber import build_huber

ridge, *_ = load_processed("../data/processed", SWEEP)
huber = build_huber(ridge.X, ridge.y, ridge.lam, "../data/processed/huber_reference.json")
pd.DataFrame([
    {"problem": "Ridge", "L": ridge.L, "mu": ridge.mu, "kappa": ridge.kappa, "f*": ridge.f_star},
    {"problem": "Huber", "L": huber.L, "mu": huber.mu, "kappa": huber.kappa, "f*": huber.f_star},
]).round(4)

huber: reference read from ../data/processed/huber_reference.json


,problem,L,mu,kappa,f*
0,Ridge,9.1156,0.0341,267.5297,6.1511
1,Huber,9.1156,0.0316,288.2609,5.1963


In [3]:
from src.experiment import HUBER_BUILDERS, ExperimentGroup, run_group
from src.figures import convergence_pair, cost_figure, save_figure

records = {}
for name, (title, build) in HUBER_BUILDERS.items():
    records[name] = run_group(ExperimentGroup(name, title, build), huber, "../results/raw")
    convergence_pair(records[name], name, title=title, out_dir="../results/figures")

huber-newton: loaded 4 runs from ../results/raw/huber-newton.json


huber-headline: loaded 5 runs from ../results/raw/huber-headline.json


In [4]:
armijo = [r for group in records.values() for r in group if "Armijo" in r.step_label]
figure = cost_figure(armijo, title="Line search cost on the Huber objective")
save_figure(figure, "huber-step_cost", "../results/figures")

[PosixPath('../results/figures/huber-step_cost.pdf'),
 PosixPath('../results/figures/huber-step_cost.png')]

## Chi phí dựng Hessian

Hessian của Huber đổi theo $w$ nên phải dựng lại ở mỗi vòng, khác với Ridge chỉ dựng một
lần. So hai cách dựng: lấy chỉ số các dòng nằm trong vùng bậc hai như `src/huber.py` cài
đặt, và nhân trọng số 0/1 vào mọi dòng rồi giữ nguyên kích thước ma trận.

In [5]:
import time

def timed(build, repeats=3):
    build()                                                  # warm up
    times = []
    for _ in range(repeats):
        t = time.perf_counter(); build(); times.append(time.perf_counter() - t)
    return float(np.median(times)) * 1e3

def masked_copy():
    """The other way to build the Huber Hessian: weight every row, keep them all."""
    s = (np.abs(huber.X @ huber.w_star - huber.y) <= huber.delta).astype(np.float64)
    weighted = huber.X * s[:, None]
    hessian = weighted.T @ huber.X / huber.n
    hessian.flat[:: huber.d + 1] += huber.lam
    return hessian

w_star = huber.w_star
outside = float(np.mean(np.abs(huber.X @ w_star - huber.y) > huber.delta))
pd.DataFrame([
    {"construction": "Ridge, all rows", "ms": timed(lambda: ridge.compute_hessian())},
    {"construction": "Huber, index selection", "ms": timed(lambda: huber.compute_hessian(w_star))},
    {"construction": "Huber, row weighting", "ms": timed(masked_copy)},
]).round(2)

,construction,ms
0,"Ridge, all rows",20.75
1,"Huber, index selection",26.27
2,"Huber, row weighting",64.06


In [6]:
# An elementwise np.allclose is the wrong comparison for this pair: dividing a
# tiny absolute error by a near-zero off-diagonal entry inflates the relative
# gap. At H[79, 113] (~-3.79e-05) the absolute difference is only 6.39e-15, but
# the elementwise relative gap it produces is 1.68e-10 -- a property of that
# denominator, not of the arithmetic. The comparison the two constructions
# actually support is the matrix-norm relative gap, which sits right where
# summing n=200,000 rows in two different orders should leave it: measured
# 1.22e-12 against a sqrt(n) * eps floor of about 9.93e-14, both around 1e-12
# to 1e-13. atol=0.0 is still passed explicitly so the tolerance below is the
# one that governs, not np.allclose's default atol=1e-8.
H_index = huber.compute_hessian(w_star)
H_weighted = masked_copy()
elementwise_relative_gap = float(np.max(np.abs(H_index - H_weighted) / np.abs(H_weighted)))
fro_relative_gap = float(np.linalg.norm(H_index - H_weighted, "fro") / np.linalg.norm(H_weighted, "fro"))
print(f"measured max elementwise relative gap (inflated by near-zero entries): {elementwise_relative_gap:.3e}")
print(f"measured Frobenius-norm relative gap: {fro_relative_gap:.3e}")
assert np.isclose(fro_relative_gap, 0.0, rtol=0.0, atol=1e-10)
print(f"fraction of rows outside the quadratic zone: {outside:.4f}")

measured max elementwise relative gap (inflated by near-zero entries): 1.683e-10
measured Frobenius-norm relative gap: 1.223e-12
fraction of rows outside the quadratic zone: 0.1960


## Ngưỡng số của gap

`suboptimality` cộng dồn theo từng dòng trước khi lấy tổng, với kỳ vọng giữ nhiều chữ số
có nghĩa hơn công thức `value(w) - f_star`. Đo trực tiếp mức sàn của từng cách tính trên
bài toán thật, đối chiếu với dạng bậc hai đúng bên trong vùng bậc hai quanh $w^*$.

In [7]:
rng = np.random.default_rng(0)
direction = rng.normal(size=huber.d)
direction /= np.linalg.norm(direction)
H_star = huber.compute_hessian(w_star)

floor_rows = []
for t in np.logspace(-1, -13, 13):
    step = t * direction
    w = w_star + step
    floor_rows.append({
        "distance": t,
        "quadratic form": 0.5 * step @ H_star @ step,
        "paired formula": huber.suboptimality(w),
        "plain difference": huber.value(w) - huber.f_star,
    })
df_floor = pd.DataFrame(floor_rows)
df_floor.map(lambda v: f"{v:.3e}")

,distance,quadratic form,paired formula,plain difference
0,1.000e-01,4.428e-03,4.428e-03,4.428e-03
1,1.000e-02,4.428e-05,4.429e-05,4.429e-05
2,1.000e-03,4.428e-07,4.428e-07,4.428e-07
3,1.000e-04,4.428e-09,4.428e-09,4.428e-09
4,1.000e-05,4.428e-11,4.428e-11,4.428e-11
5,1.000e-06,4.428e-13,4.430e-13,4.414e-13
6,1.000e-07,4.428e-15,4.543e-15,4.441e-15
7,1.000e-08,4.428e-17,-1.317e-16,-1.776e-15
8,1.000e-09,4.428e-19,5.349e-17,0.000e+00
9,1.000e-10,4.428e-21,1.046e-16,-8.882e-16


In [8]:
from src.experiment import summary_table
pd.DataFrame(summary_table(records["huber-headline"])).round(6)

,method,configuration,status,iters_to_1e-06,seconds_to_1e-06,final_gap,iterations_run,total_seconds,fevals_per_iter,epochs
0,GD,t = 1.9/L,converged,381.0,5.439562,-0.000000,2108,30.643619,0.000000,NaN
1,GD,"Armijo (c = 0.3, rho = 0.5, t0 = 1)",stalled,220.0,4.826465,0.000000,1051,27.214633,4.200761,NaN
2,"AGD (beta from t, mu) + restart",t = 1/L,converged,83.0,1.193137,-0.000000,301,4.290277,0.000000,NaN
3,SGD (B = 2048),eta = 0.00936,max_iter,NaN,NaN,0.000313,3880,2.273806,0.000000,39.7312
4,Newton,t = 1,converged,7.0,0.341344,0.000000,8,0.382789,0.000000,NaN


In [9]:
newton = next(r for r in records["huber-newton"] if r.label.startswith("Newton,") and "reused" not in r.label)
pd.DataFrame({"iteration": newton.iters, "gap": newton.gaps}).head(12)

,iteration,gap
0,0,3.847523e+00
1,1,2.837399e+00
2,2,1.195442e+00
3,3,6.174399e-01
4,4,1.928448e-01
5,5,1.100443e-02
6,6,1.977841e-05
7,7,6.654556e-11
8,8,1.656476e-17
